In [0]:
%sql
use catalog capstone_team1;
use schema silver;

In [0]:
primary_df=spark.table("capstone_team1.bronze.bronze_primary_events")
secondary_df=spark.table("capstone_team1.bronze.bronze_secondary_events")
status_df=spark.table("capstone_team1.bronze.bronze_status_updates")
primary_df.printSchema()
secondary_df.printSchema()
status_df.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType
from pyspark.sql import functions as F

# Define explicit schemas
primary_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("ingest_timestamp", StringType(), True),
    StructField("merchant_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("payment_channel", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("risk_score", DoubleType(), True),
    StructField("device_id", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("source_schema_version", StringType(), True)
])

secondary_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("transaction_id", StringType(), True),
    StructField("event_timestamp", TimestampType(), True),
    StructField("related_entity_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_value", DoubleType(), True),
    StructField("source_system", StringType(), True)
])

status_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("change_timestamp", TimestampType(), True),
    StructField("previous_status", StringType(), True),
    StructField("new_status", StringType(), True),
    StructField("change_reason", StringType(), True),
    StructField("source_sequence", IntegerType(), True),
    StructField("source_system", StringType(), True)
])

# Cast all columns in each DataFrame based on the explicit schemas
def cast_to_schema(df, schema):
    for field in schema.fields:
        if field.name in df.columns:
            df = df.withColumn(field.name, F.col(field.name).cast(field.dataType))
    return df

primary_df = cast_to_schema(primary_df, primary_schema)
secondary_df = cast_to_schema(secondary_df, secondary_schema)
status_df = cast_to_schema(status_df, status_schema)

# Verify the casted schemas
print("Primary Events Schema:")
primary_df.printSchema()
print("Secondary Events Schema:")
secondary_df.printSchema()
print("Status Updates Schema:")
status_df.printSchema()

In [0]:
print("Primary events")
print(primary_df.columns)
print("\nSecondary events")
print(secondary_df.columns)
print("\nStatus updates")
print(status_df.columns)    

In [0]:
# Data Quality Rule Catalogue

dq_rules = [
    # Primary Events
    ("primary_events", "transaction_id", "NOT NULL"),
    ("primary_events", "event_timestamp", "VALID TIMESTAMP"),
    ("primary_events", "ingest_timestamp", "VALID TIMESTAMP"),
    ("primary_events", "merchant_id", "NOT NULL"),
    ("primary_events", "customer_id", "NOT NULL"),
    ("primary_events", "amount", "MUST BE POSITIVE"),
    ("primary_events", "risk_score", "BETWEEN 0 AND 100"),
    ("primary_events", "amount", "NO EXTREME VALUES (|Z-SCORE| <= 3)"),
    ("primary_events", "payment_status", "NOT NULL"),
    ("primary_events", "currency", "NOT NULL"),
    ("primary_events", "payment_channel", "NOT NULL"),
    ("primary_events", "device_id", "NOT NULL"),
    ("primary_events", "all string columns", "NO BLANK OR PLACEHOLDER VALUES"),
    ("primary_events", "ingest_timestamp", "INGEST >= EVENT TIMESTAMP"),
    ("primary_events", "all source columns", "NO EXACT DUPLICATE RECORDS"),
    ("primary_events", "transaction_id", "UNIQUE (KEEP LATEST INGEST)"),

    # Secondary Events
    ("secondary_events", "event_id", "NOT NULL"),
    ("secondary_events", "transaction_id", "NOT NULL"),
    ("secondary_events", "event_timestamp", "VALID TIMESTAMP"),
    ("secondary_events", "event_type", "NOT NULL"),
    ("secondary_events", "related_entity_id", "NOT NULL"),
    ("secondary_events", "event_value", "NOT NULL"),
    ("secondary_events", "all string columns", "NO BLANK OR PLACEHOLDER VALUES"),
    ("secondary_events", "all source columns", "NO EXACT DUPLICATE RECORDS"),
    ("secondary_events", "event_id", "UNIQUE"),

    # Status Updates
    ("status_updates", "transaction_id", "NOT NULL"),
    ("status_updates", "change_timestamp", "VALID TIMESTAMP"),
    ("status_updates", "new_status", "NOT NULL"),
    ("status_updates", "change_reason", "NOT NULL"),
    ("status_updates", "previous_status", "NOT NULL"),
    ("status_updates", "source_sequence", "NOT NULL"),
    ("status_updates", "all string columns", "NO BLANK OR PLACEHOLDER VALUES"),
    ("status_updates", "all source columns", "NO EXACT DUPLICATE RECORDS"),
    ("status_updates", "transaction_id + source_sequence", "UNIQUE"),
]

# Convert rules into a DataFrame
dq_catalogue = spark.createDataFrame(
    dq_rules,
    ["dataset", "column_name", "dq_rule"]
)

display(dq_catalogue)

In [0]:
#verify Catalogue
print("Total DQ Rules:", dq_catalogue.count())
dq_catalogue.groupBy("dataset").count().show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Text values treated as "missing" in addition to NULL
PLACEHOLDERS = ["", "na", "n/a", "null", "none", "nan", "-", "?"]
# Ingestion metadata columns are ignored when comparing records for duplicates
METADATA_COLS = ["_rescued_data", "_source_file", "_ingestion_timestamp"]

# Timestamp formats seen in the primary_events files (2-digit year tried before 4-digit)
TS_FORMATS = [
    "yyyy-MM-dd'T'HH:mm:ss",
    "yyyy-MM-dd HH:mm:ss",
    "M/d/yy H:mm",
    "M/d/yy h:mm:ss a",
    "M/d/yyyy H:mm",
    "M/d/yyyy h:mm:ss a",
]

def parse_timestamp(c):
    """Try each known format; NULL if none match (e.g. '4:09:60 AM')."""
    return F.coalesce(*[F.try_to_timestamp(F.col(c), F.lit(f)) for f in TS_FORMATS])

def missing_checks(df, cols):
    """One DQ reason per column: NULL, or blank / placeholder text for string columns."""
    dtypes = dict(df.dtypes)
    checks = []
    for c in cols:
        cond = F.col(c).isNull()
        if dtypes.get(c) == "string":
            cond = cond | F.lower(F.trim(F.col(c))).isin(PLACEHOLDERS)
        checks.append(F.when(cond, F.lit(f"Missing {c}")))
    return checks

def add_duplicate_flags(df, key_cols, order_cols):
    """
    _exact_dup_rn > 1 : identical copy of another record (all source columns equal)
    _key_dup_rn  > 1  : a different version with the same business key ranks higher (order_cols)
    dense_rank gives identical copies the same key rank, so the kept copy never depends on
    which of two identical rows row_number happened to pick.
    """
    source_cols = [c for c in df.columns if c not in METADATA_COLS]
    w_exact = Window.partitionBy(*source_cols).orderBy(F.col("_ingestion_timestamp").asc_nulls_last())
    # hash of the source columns is a deterministic tie-break between different versions
    w_key = Window.partitionBy(*key_cols).orderBy(*order_cols, F.xxhash64(*source_cols))
    return df.withColumns({
        "_exact_dup_rn": F.row_number().over(w_exact),
        "_key_dup_rn": F.dense_rank().over(w_key),
    })

def duplicate_checks(key_cols):
    key_label = " + ".join(key_cols)
    key_present = F.lit(True)
    for k in key_cols:
        key_present = key_present & F.col(k).isNotNull()
    return [
        F.when(F.col("_exact_dup_rn") > 1, F.lit("Duplicate record")),
        F.when((F.col("_key_dup_rn") > 1) & key_present,
               F.lit(f"Duplicate {key_label}")),
    ]

def split_valid_invalid(checked_df):
    checked_df = checked_df.drop("_exact_dup_rn", "_key_dup_rn")
    valid = checked_df.filter(F.col("dq_reason") == "")
    invalid = checked_df.filter(F.col("dq_reason") != "")
    return checked_df, valid, invalid

def dq_reason_summary(invalid_df):
    """Count of records per individual DQ reason (a record can have several)."""
    return (invalid_df
            .select(F.explode(F.split(F.col("dq_reason"), ", ")).alias("dq_reason"))
            .groupBy("dq_reason").count()
            .orderBy(F.col("count").desc()))

In [0]:
# Compare extreme-value definitions for amount and risk_score before choosing the DQ threshold
num_cols = ["amount", "risk_score"]
stats = primary_df.agg(*[F.mean(c).alias(f"{c}_mean") for c in num_cols],
                       *[F.stddev(c).alias(f"{c}_std") for c in num_cols]).first()

rows = []
for c in num_cols:
    q1, q3 = primary_df.approxQuantile(c, [0.25, 0.75], 0.0001)
    iqr = q3 - q1
    m, s = stats[f"{c}_mean"], stats[f"{c}_std"]
    rules = {f"IQR x{k}": (q1 - k * iqr, q3 + k * iqr) for k in [1.5, 3, 5, 10]}
    rules.update({f"z-score > {z}": (m - z * s, m + z * s) for z in [3, 4, 5]})
    for rule, (lo, hi) in rules.items():
        rows.append((c, rule, float(lo), float(hi)))

thresholds = spark.createDataFrame(rows, "column string, rule string, lower double, upper double")

# Count rows outside each threshold in one pass per column
counts = []
for c in num_cols:
    t = thresholds.filter(F.col("column") == c)
    counts.append(primary_df.select(c).crossJoin(t)
                  .groupBy("column", "rule", "lower", "upper")
                  .agg(F.count(F.when((F.col(c) < F.col("lower")) | (F.col(c) > F.col("upper")), 1)).alias("rows_outside")))
display(counts[0].unionByName(counts[1]).orderBy("column", "rows_outside"))

display(primary_df.select("transaction_id", "amount", "risk_score").orderBy(F.col("amount").desc()).limit(50))

In [0]:
# Validate Primary Events
from pyspark.sql.functions import *

# Parse the mixed-format timestamp strings
primary_parsed_df = primary_df.withColumns({
    "event_ts": parse_timestamp("event_timestamp"),
    "ingest_ts": parse_timestamp("ingest_timestamp"),
})

# Duplicate flags: exact copies, and repeated transaction_id (keep the latest ingest)
primary_parsed_df = add_duplicate_flags(
    primary_parsed_df,
    key_cols=["transaction_id"],
    order_cols=[F.col("ingest_ts").desc_nulls_last()],
)

# Extreme amount threshold: more than 3 standard deviations from the mean
Z_THRESHOLD = 3
amount_stats = primary_df.agg(F.mean("amount").alias("mean"), F.stddev("amount").alias("std")).first()
amount_lower = amount_stats["mean"] - Z_THRESHOLD * amount_stats["std"]
amount_upper = amount_stats["mean"] + Z_THRESHOLD * amount_stats["std"]
print(f"Extreme amount bounds (|z| > {Z_THRESHOLD}): < {amount_lower:.2f} or > {amount_upper:.2f}")

primary_required = ["transaction_id", "merchant_id", "customer_id", "payment_channel",
                    "payment_status", "currency", "device_id", "event_timestamp", "ingest_timestamp"]

# Identify invalid primary event records (existing rules + missing/duplicate rules)
primary_checked_df = primary_parsed_df.withColumn(
    "dq_reason",
    concat_ws(
        ", ",
        *missing_checks(primary_parsed_df, primary_required),
        when(col("event_timestamp").isNotNull() & col("event_ts").isNull(), "Invalid event_timestamp"),
        when(col("ingest_timestamp").isNotNull() & col("ingest_ts").isNull(), "Invalid ingest_timestamp"),
        when(col("ingest_ts") < col("event_ts"), "Ingest before event"),
        when(col("amount").isNull() | (col("amount") <= 0), "Invalid amount"),
        when(
            col("risk_score").isNull() |
            (col("risk_score") < 0) |
            (col("risk_score") > 100),
            "Invalid risk_score"
        ),
        when((col("amount") < amount_lower) | (col("amount") > amount_upper), "Extreme amount"),
        *duplicate_checks(["transaction_id"]),
    )
)

# Separate valid and invalid records
primary_checked_df, valid_primary_df, quarantine_primary_df = split_valid_invalid(primary_checked_df)

print("Total records:", primary_df.count())
print("Valid records:", valid_primary_df.count())
print("Invalid records:", quarantine_primary_df.count())
print("Distinct transaction_id in valid:", valid_primary_df.select("transaction_id").distinct().count())
display(dq_reason_summary(quarantine_primary_df))
display(quarantine_primary_df)

In [0]:
# Validate Secondary Events

# Duplicate flags: exact copies, and repeated event_id (latest event_timestamp kept)
secondary_flagged_df = add_duplicate_flags(
    secondary_df,
    key_cols=["event_id"],
    order_cols=[F.col("event_timestamp").desc_nulls_last()],
)

secondary_required = ["event_id", "transaction_id", "event_timestamp", "event_type",
                      "related_entity_id", "event_value", "source_system"]

secondary_checked_df = secondary_flagged_df.withColumn(
    "dq_reason",
    concat_ws(
        ", ",
        *missing_checks(secondary_flagged_df, secondary_required),
        *duplicate_checks(["event_id"]),
    )
)

# Separate valid and invalid records
secondary_checked_df, valid_secondary_df, quarantine_secondary_df = split_valid_invalid(secondary_checked_df)
print("Total records:", secondary_df.count())
print("Valid records:", valid_secondary_df.count())
print("Invalid records:", quarantine_secondary_df.count())

display(dq_reason_summary(quarantine_secondary_df))
display(quarantine_secondary_df)

In [0]:
# Validate Status Updates

# Duplicate flags: exact copies, and repeated (transaction_id, source_sequence).
# A transaction can legitimately have several status changes, so transaction_id alone is not unique.
status_flagged_df = add_duplicate_flags(
    status_df,
    key_cols=["transaction_id", "source_sequence"],
    order_cols=[F.col("change_timestamp").desc_nulls_last()],
)

status_required = ["transaction_id", "change_timestamp", "previous_status", "new_status",
                   "change_reason", "source_sequence", "source_system"]

status_checked_df = status_flagged_df.withColumn(
    "dq_reason",
    concat_ws(
        ", ",
        *missing_checks(status_flagged_df, status_required),
        *duplicate_checks(["transaction_id", "source_sequence"]),
    )
)

# Separate valid and invalid records
status_checked_df, valid_status_df, quarantine_status_df = split_valid_invalid(status_checked_df)

print("Total records:", status_df.count())
print("Valid records:", valid_status_df.count())
print("Invalid records:", quarantine_status_df.count())

display(dq_reason_summary(quarantine_status_df))
display(quarantine_status_df)

In [0]:
# Reconcile duplicate checks for every dataset:
#   expected_dup_rows = rows that share a business key with another row (all but one per key)
#   flagged_dup_rows  = quarantined rows carrying a "Duplicate ..." reason
#   keys_without_valid_row = keys with no valid record left (should only fail non-duplicate rules)
reconcile_sets = [
    # (dataset, source df, valid df, quarantine df, business key)
    ("primary_events",   primary_df,   valid_primary_df,   quarantine_primary_df,   ["transaction_id"]),
    ("secondary_events", secondary_df, valid_secondary_df, quarantine_secondary_df, ["event_id"]),
    ("status_updates",   status_df,    valid_status_df,    quarantine_status_df,    ["transaction_id", "source_sequence"]),
]

summary_frames, group_frames, lost_frames = [], [], []

for name, src_df, valid_df, quarantine_df, keys in reconcile_sets:
    src_cols = [c for c in src_df.columns if c not in METADATA_COLS]
    key_label = " + ".join(keys)
    keyed_src = src_df.dropna(subset=keys)

    # Shape of repeated key groups: how many rows and how many distinct versions
    groups = (keyed_src.groupBy(*keys)
              .agg(F.count("*").alias("rows"),
                   F.countDistinct(F.struct(*src_cols)).alias("distinct_versions"))
              .filter("rows > 1"))
    group_frames.append(
        groups.groupBy("rows", "distinct_versions").agg(F.count("*").alias("keys"))
              .withColumn("expected_flagged_rows", (F.col("rows") - 1) * F.col("keys"))
              .select(F.lit(name).alias("dataset"), F.lit(key_label).alias("business_key"), "*"))

    # Keys that ended up with no valid record at all
    lost_keys = keyed_src.select(*keys).distinct().join(valid_df.select(*keys).distinct(), keys, "left_anti")
    lost = quarantine_df.join(lost_keys, keys)
    lost_frames.append(lost.select(
        F.lit(name).alias("dataset"),
        F.concat_ws(" | ", *[F.col(k).cast("string") for k in keys]).alias("business_key_value"),
        "dq_reason"))

    expected = keyed_src.groupBy(*keys).count().agg(F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("expected_dup_rows"))
    flagged = quarantine_df.filter(F.col("dq_reason").contains("Duplicate")).agg(F.count("*").alias("flagged_dup_rows"))
    summary_frames.append(
        expected.crossJoin(flagged)
                .crossJoin(lost_keys.agg(F.count("*").alias("keys_without_valid_row")))
                .crossJoin(valid_df.agg(F.count("*").alias("valid_rows"),
                                        F.countDistinct(*keys).alias("distinct_keys_in_valid")))
                .select(F.lit(name).alias("dataset"), F.lit(key_label).alias("business_key"), "*"))

def union_all(frames):
    out = frames[0]
    for f in frames[1:]:
        out = out.unionByName(f)
    return out

reconcile_summary = (union_all(summary_frames)
    .withColumn("dup_check_ok", (F.col("expected_dup_rows") == F.col("flagged_dup_rows")) &
                                (F.col("valid_rows") == F.col("distinct_keys_in_valid"))))

print("Duplicate reconciliation summary")
display(reconcile_summary)
print("Repeated key group shapes")
display(union_all(group_frames).orderBy("dataset", "rows", "distinct_versions"))
print("Keys with no valid record")
display(union_all(lost_frames).orderBy("dataset", "business_key_value"))

In [0]:
display(
    primary_df
    .groupBy("transaction_id")
    .count()
    .filter(col("count") > 1)
    .orderBy("transaction_id")
)

In [0]:
display(
    primary_df
    .filter(
        col("transaction_id").between(
            "TRANSACTION-0000501",
            "TRANSACTION-0000540"
        )
    )
    .orderBy("transaction_id", "_ingestion_timestamp")
)

In [0]:
from pyspark.sql.functions import *

duplicate_ids = (
    primary_df
    .groupBy("transaction_id")
    .count()
    .filter(col("count") > 1)
    .select("transaction_id")
)

duplicate_records = primary_df.join(
    duplicate_ids,
    "transaction_id",
    "inner"
)

display(
    duplicate_records
    .orderBy("transaction_id", "_source_file")
)

In [0]:
# Deduplicate Primary Events

primary_dedup_df = primary_df.dropDuplicates(
    ["transaction_id"]
)

print("Before deduplication:", primary_df.count())
print("After deduplication:", primary_dedup_df.count())
print(
    "Removed duplicate rows:",
    primary_df.count() - primary_dedup_df.count()
)

In [0]:
secondary_dedup_df = secondary_df.dropDuplicates(
    ["event_id"]
)

print("Before:", secondary_df.count())
print("After:", secondary_dedup_df.count())

status_dedup_df = status_df.dropDuplicates(
    ["transaction_id", "source_sequence"]
)

print("Before:", status_df.count())
print("After:", status_dedup_df.count())

print("PRIMARY:", primary_dedup_df.count())
print("SECONDARY:", secondary_dedup_df.count())
print("STATUS:", status_dedup_df.count())

CDC

In [0]:
# Display status update history
display(
    status_dedup_df
    .orderBy(
        "transaction_id",
        "source_sequence"
    )
)
# Check status distribution
status_dedup_df.groupBy("new_status") \
    .count() \
    .orderBy("new_status") \
    .show()
# Check sequence ordering
status_dedup_df.select(
    "transaction_id",
    "source_sequence",
    "change_timestamp",
    "previous_status",
    "new_status"
).orderBy(
    "transaction_id",
    "source_sequence"
).show(20, truncate=False)


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
status_window = Window.partitionBy(
    "transaction_id"
).orderBy(
    col("source_sequence").desc(),
    col("change_timestamp").desc()
)
latest_status_df = (
    status_dedup_df
    .withColumn("row_num", row_number().over(status_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)
display(
    latest_status_df.select(
        "transaction_id",
        "source_sequence",
        "previous_status",
        "new_status",
        "change_timestamp",
        "change_reason"
    )
)
print("Total status history:", status_dedup_df.count())
print("Transactions with latest status:", latest_status_df.count())

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Read deduplicated Status Updates
status_df = status_dedup_df

# 1. Preserve complete status history
status_history_df = status_df

# 2. Identify the latest status using source_sequence
status_window = Window.partitionBy(
    "transaction_id"
).orderBy(
    col("source_sequence").desc(),
    col("change_timestamp").desc()
)

latest_status_df = (
    status_history_df
    .withColumn("row_num", row_number().over(status_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

# 3. Check status transition consistency
transition_window = Window.partitionBy(
    "transaction_id"
).orderBy("source_sequence")

status_history_df = (
    status_history_df
    .withColumn(
        "expected_previous_status",
        lag("new_status").over(transition_window)
    )
    .withColumn(
        "transition_check",
        when(
            col("expected_previous_status").isNull(),
            "FIRST EVENT"
        ).when(
            col("previous_status") == col("expected_previous_status"),
            "VALID"
        ).otherwise("INVALID")
    )
)

# 4. Display complete status history
print("COMPLETE STATUS HISTORY")
display(status_history_df.orderBy("transaction_id", "source_sequence"))

# 5. Display latest status
print("LATEST STATUS PER TRANSACTION")
display(latest_status_df)

# 6. Summary
print("Total status history:", status_history_df.count())
print("Transactions with latest status:", latest_status_df.count())

status_history_df.groupBy("transition_check").count().show()

In [0]:
from pyspark.sql.functions import *

# 1. Select transaction amount from Primary Events
primary_amount_df = primary_dedup_df.select(
    "transaction_id",
    col("amount").alias("primary_amount")
)

# 2. Convert event_value to numeric
secondary_amount_df = (
    secondary_dedup_df
    .withColumn("event_value", col("event_value").cast("double"))
    .groupBy("transaction_id")
    .agg(
        sum("event_value").alias("secondary_amount")
    )
)

# 3. Join both datasets
reconciliation_df = primary_amount_df.join(
    secondary_amount_df,
    "transaction_id",
    "left"
)

# 4. Compare amounts
reconciliation_df = reconciliation_df.withColumn(
    "amount_difference",
    col("primary_amount") - col("secondary_amount")
)

reconciliation_df = reconciliation_df.withColumn(
    "reconciliation_status",
    when(
        col("secondary_amount").isNull(),
        "NO SECONDARY EVENT"
    ).when(
        abs(col("amount_difference")) < 0.01,
        "MATCHED"
    ).otherwise("MISMATCH")
)

# 5. Display reconciliation results
display(reconciliation_df)

# 6. Summary
reconciliation_df.groupBy("reconciliation_status") \
    .count() \
    .show()

In [0]:
# UNIT TESTS FOR FRAUD WATCH

# 1. Test Primary Events deduplication
assert primary_dedup_df.count() == 11500, "Primary deduplication failed"
assert (
    primary_dedup_df
    .groupBy("transaction_id")
    .count()
    .filter(col("count") > 1)
    .count() == 0
), "Duplicate transaction IDs found in Primary Events"


# 2. Test Secondary Events deduplication
assert (
    secondary_dedup_df
    .groupBy("event_id")
    .count()
    .filter(col("count") > 1)
    .count() == 0
), "Duplicate event IDs found in Secondary Events"


# 3. Test Status Updates deduplication
assert (
    status_dedup_df
    .groupBy("transaction_id", "source_sequence")
    .count()
    .filter(col("count") > 1)
    .count() == 0
), "Duplicate status records found"


# 4. Test latest status
assert (
    latest_status_df
    .groupBy("transaction_id")
    .count()
    .filter(col("count") > 1)
    .count() == 0
), "Multiple latest statuses found for a transaction"


# 5. Test reconciliation
assert "reconciliation_status" in reconciliation_df.columns
assert "amount_difference" in reconciliation_df.columns

assert (
    reconciliation_df
    .filter(col("reconciliation_status").isNull())
    .count() == 0
), "Null reconciliation status found"


print("All unit tests passed!")